# CryoLens — Sentinel-1 EW Subswath Thermal Noise & Scalloping Comparison

**Objective:** Demonstrate the impact of inter-subswath thermal noise removal (`s1denoise` Park et al. algorithm) on Sentinel-1 Extra Wide (EW) Swath HV cross-polarization data compared to standard calibration.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from cryolens.preprocess.s1denoise import S1SubswathDenoise

# 1. Simulate Sentinel-1 EW cross-pol HV swath (5 subswaths: EW1 - EW5)
h, w = 400, 1000
np.random.seed(42)

# Calm ocean backscatter in linear intensity (~ -32 dB = 0.00063)
ocean_linear = np.random.exponential(scale=0.00063, size=(h, w)).astype(np.float32)

# Add NESZ scalloping patterns per subswath (oscillating -28 dB to -24 dB)
x = np.linspace(-1, 1, w)
# Parabolic beam pattern modulation across each of the 5 subswaths
swath_width = w // 5
nesz_profile = np.zeros(w, dtype=np.float32)
for i in range(5):
    sub_x = np.linspace(-1, 1, swath_width)
    nesz_db = -28.0 + 3.5 * (sub_x**2)
    nesz_profile[i * swath_width : (i + 1) * swath_width] = 10.0 ** (nesz_db / 10.0)

noisy_hv_linear = ocean_linear + np.tile(nesz_profile, (h, 1))

# Add two iceberg targets in EW2 and EW4
noisy_hv_linear[180:186, 280:286] = 0.08  # ~ -11 dB
noisy_hv_linear[220:226, 680:686] = 0.06  # ~ -12 dB

# 2. Apply s1denoise inter-subswath power balancing
denoiser = S1SubswathDenoise()
denoised_hv_linear, factors = denoiser.denoise(noisy_hv_linear)

# Convert to Decibels
raw_hv_db = 10.0 * np.log10(np.maximum(noisy_hv_linear, 1e-5))
denoised_hv_db = 10.0 * np.log10(np.maximum(denoised_hv_linear, 1e-5))

print("Computed Subswath Balance Factors:", factors)

In [ ]:
# 3. Plot Cross-Swath Range Profiles (Azimuth Average)
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Profile 1: Range Profile in dB
raw_profile_db = np.mean(raw_hv_db, axis=0)
denoised_profile_db = np.mean(denoised_hv_db, axis=0)

axes[0].plot(
    raw_profile_db, label="Raw Calibrated HV (with NESZ Scalloping)", color="crimson", alpha=0.8
)
axes[0].plot(
    denoised_profile_db, label="s1denoise Corrected HV (Park et al.)", color="teal", linewidth=2
)
axes[0].set_ylabel(r"Mean $\sigma^0_{HV}$ (dB)", fontsize=12)
axes[0].set_title(
    "Sentinel-1 EW Range Profile: HV Subswath Scalloping Removal", fontsize=14, fontweight="bold"
)
axes[0].grid(True, linestyle="--", alpha=0.5)
axes[0].legend(fontsize=11)

# Draw subswath boundaries
for i in range(1, 5):
    axes[0].axvline(
        i * swath_width, color="gray", linestyle=":", label="Subswath Boundary" if i == 1 else ""
    )
    axes[0].text(
        i * swath_width - swath_width // 2,
        -23,
        f"EW{i}",
        ha="center",
        fontsize=11,
        fontweight="bold",
        color="navy",
    )
axes[0].text(
    5 * swath_width - swath_width // 2,
    -23,
    "EW5",
    ha="center",
    fontsize=11,
    fontweight="bold",
    color="navy",
)

# Profile 2: Image Difference Slice
diff_profile = raw_profile_db - denoised_profile_db
axes[1].plot(diff_profile, color="indigo", label="Estimated Subswath Noise Floor Removal (dB)")
axes[1].set_xlabel("Range Pixel (Cross-Track Direction)", fontsize=12)
axes[1].set_ylabel("Subtracted Clutter (dB)", fontsize=12)
axes[1].grid(True, linestyle="--", alpha=0.5)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()